# QA Chatbot Demo — Strands Agents SDK

A simple interactive question-answering chatbot built with [Strands Agents](https://github.com/strands-agents/sdk-python).

It:
- Keeps conversation history automatically across turns (the `Agent` object is stateful).
- Has a small FAQ knowledge-base tool it checks before falling back to general knowledge.
- Can be run turn-by-turn in notebook cells, or via an interactive loop at the end.

**Requires model credentials** (e.g. AWS Bedrock access, or an Anthropic API key configured for the Strands `AnthropicModel` provider) to actually generate responses.

## 1. Install the SDK

In [ ]:
%pip install strands-agents ollama

## 2. Define a small FAQ knowledge base and a search tool

In [ ]:
from strands import Agent, tool

FAQ_DB = {
    "what is strands": (
        "Strands Agents is an open-source Python SDK for building AI agents "
        "that can reason, call tools, and hold multi-turn conversations."
    ),
    "how do i install strands": (
        "Run 'pip install strands-agents' in your Python environment."
    ),
    "what model providers does strands support": (
        "Strands supports multiple model providers, including Amazon Bedrock "
        "and Anthropic's API, among others."
    ),
    "how do i reset the conversation": (
        "Create a new Agent instance, or call agent.messages.clear() to wipe "
        "its conversation history."
    ),
}


@tool
def search_faq(query: str) -> str:
    """Search the internal FAQ knowledge base for a relevant answer.

    Args:
        query: The user's question, used to look up a matching FAQ entry.
    """
    query_lower = query.lower().strip("? ")
    for key, answer in FAQ_DB.items():
        if key in query_lower or query_lower in key:
            return answer
    return "No matching FAQ entry found. Answer using general knowledge instead."


### Check which models you already have pulled

Run this first. If it errors with a connection error, Ollama isn't running — start it (`ollama serve`, or just open the Ollama app). If the list is empty or doesn't include the model you want, pull one:

```bash
ollama pull llama3.1
```

In [ ]:
import ollama

try:
    models = ollama.list()
    names = [m.model for m in models.models]
    print("Models available locally:", names if names else "(none pulled yet)")
except Exception as e:
    print("Could not reach Ollama:", e)
    print("Make sure Ollama is installed and running (ollama serve), then re-run this cell.")


## 3. Build the QA agent (using a free local model via Ollama)

This uses [Ollama](https://ollama.com) to run a model **locally for free** — no API key, no cloud account.

**One-time setup, outside this notebook:**
1. Install Ollama: https://ollama.com/download
2. Pull a small model, e.g.:
   ```bash
   ollama pull llama3.2:1b
   ```
   (`llama3.2:1b` is ~1.3GB — the smallest in the Llama family, good for quick testing. `llama3.2:3b` is a step up in quality if 1b struggles with tool-calling.)
3. Make sure the Ollama server is running (it starts automatically after install, or run `ollama serve`).

That's it — the cell below points Strands at your local Ollama server (`http://localhost:11434` by default).

In [ ]:
from strands.models.ollama import OllamaModel

SYSTEM_PROMPT = """You are a concise, helpful QA chatbot.
Always check the FAQ knowledge base first using the search_faq tool.
If it returns a relevant answer, use it. Otherwise, answer from your own
knowledge, and say so clearly. Keep answers short and direct.
"""

# Points at a local Ollama server. llama3.2:1b is small and fast (~1.3GB).
# Bump to "llama3.2:3b" if 1b struggles with tool-calling, or run `ollama list`
# to see other models you've pulled.
model = OllamaModel(
    host="http://localhost:11434",
    model_id="llama3.2:1b",
)

agent = Agent(
    system_prompt=SYSTEM_PROMPT,
    tools=[search_faq],
    model=model,
)

print("Agent ready. Registered tools:", agent.tool_names)


## 4. Ask a few questions

In [ ]:
response = agent("What is Strands?")
print(response)


In [ ]:
response = agent("Does it remember what I just asked?")
print(response)


## 5. Optional: interactive chat loop\n\nRun this cell to chat turn-by-turn. Type `quit` or `exit` to stop.

In [ ]:
def chat():
    print("QA Chatbot (Strands Agents demo). Type 'quit' or 'exit' to stop.\n")
    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in {"quit", "exit"}:
            print("Goodbye!")
            break
        if not user_input:
            continue
        response = agent(user_input)
        print(f"Bot: {response}\n")

chat()
